### Fetching Data From Ambitionbox Webpage

In [22]:
import pandas as pd 
import requests
from bs4 import BeautifulSoup  #Library Used for Webscraping

In [23]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer": "https://www.google.com/",
    "DNT": "1",
}

# Headers Used for sending request to Webpage for getting access

In [28]:
all_df = []
Total_page = 300
for page in range(1, Total_page+1):

    url = f"https://www.ambitionbox.com/list-of-companies?campaign=homepage_companies_widget&page={page}"

    response = requests.get(url, headers=headers)
    webpage = response.text

    soup = BeautifulSoup(webpage, "lxml")

    company = soup.find_all("div", class_="companyCardWrapper")
    '''print(f"Page {j}: {len(company)} companies found")'''

    name = []
    rating = []
    review = []
    company_type = []
    headquarters = []
    other_locations = []

    for comapany_card in company:

        name.append(comapany_card.find("h2").text.strip())

        rating_tag = comapany_card.find("div", class_="rating_text rating_text--md")

        # Check if the rating exists because some companies may not have a rating.
        # If the tag is missing, store None to keep all lists the same length.
        if rating_tag:
            rating.append(rating_tag.text.strip())
        else:
            rating.append(None)

        # Check if review count is available.
        # Missing review information is stored as None.
        review_tag = comapany_card.find("span", class_="companyCardWrapper__companyRatingCount")
        review.append(review_tag.text.strip() if review_tag else None)

        info = comapany_card.find(
            "span",
            class_="companyCardWrapper__interLinking"
        ).text.strip()

        company_type.append(info.split("|")[0].strip())

        headquarters.append(
            info.split("+")[0].strip()
        )

        other_locations.append(
            info.split("+")[1].strip() if "+" in info else None
        )

    d = {
        "Name": name,
        "Rating": rating,
        "Review": review,
        "Company Type": company_type,
        "Headquarters": headquarters,
        "Other Locations": other_locations
    }

    df = pd.DataFrame(d)
    all_df.append(df)

# Combine all pages
final = pd.concat(all_df, ignore_index=True)


In [29]:
# Combining data of all pages 
final.head()


,Name,Rating,Review,Company Type,Headquarters,Other Locations
0,TCS,3.3,(1.2L),IT Services & Consulting,IT Services & Consulting | Bengaluru,476 other locations
1,Accenture,3.7,(75.6k),IT Services & Consulting,IT Services & Consulting | Bengaluru,279 other locations
2,Wipro,3.6,(66.6k),IT Services & Consulting,IT Services & Consulting | Hyderabad,387 other locations
3,Cognizant,3.7,(63k),IT Services & Consulting,IT Services & Consulting | Hyderabad,252 other locations
4,Capgemini,3.6,(55.1k),IT Services & Consulting,IT Services & Consulting | Bengaluru,204 other locations


In [30]:
final.to_csv("AmbitionBox_Companies.csv", index=False)